# Lab 4: Writing the Description Prompt

**Workshop 2, block 3. 20 minutes.**

Whatever the description does not mention is unavailable to retrieval.
This prompt sets the upper limit on your Level 3 and Level 5 scores.

1. Write your prompt
2. Run it over the five images and **read the output before indexing**
3. Revise and run again
4. Index the descriptions and re-score the Lv3 questions
5. Swap prompts with the team next to you

Step 2 is the one people skip. If you cannot answer the test questions
from the descriptions, neither can your index.

> **Before you start:** `data/chroma` from lab 2, `images/` from the workshop folder, and ideally `data/images.json` from lab 3.
>
> **When you finish:** `data/descriptions.json`, and image chunks added to `data/chroma` tagged `kind="image"`. Lab 5 filters on that tag.

---
## Setup

Run these two cells first. They are identical in every lab, so each
notebook works on its own.

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops. **Do not commit a notebook with a key
visible in its output.**

In [3]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

chat    OK      Linked
vision  OK      connected
embed   OK      1536 dimensions


In [9]:
# ---- helpers ------------------------------------------------------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# One PersistentClient per path, cached for the life of this kernel.
# chromadb caches internal state per path, so deleting the folder and
# opening a fresh PersistentClient while an earlier one from this same
# session is still alive corrupts the connection: you get "attempt to
# write a readonly database" or "database is locked" on the very next
# call. Rebuilding your index more than once per session, which the
# "change one setting, re-run" loop asks you to do, hits this every
# time with the naive version.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P

    if path not in _stores:
        # First time this path is opened in this session. Safe to wipe a
        # stale, wrong-chromadb-version index here, since no client for
        # this path exists in this process yet.
        if reset and _P(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by "
                f"a different chromadb version. Delete that folder and rebuild, or "
                f"install the pinned version from requirements.txt."
            ) from None

    client = _stores[path]
    if reset:
        # Reset now means delete-and-recreate the COLLECTION on the same
        # client, not delete-and-recreate the DIRECTORY under it. This is
        # what actually avoids the readonly/locked error on every rebuild
        # after the first.
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

helpers loaded


---
## The five fields

| Field | Ask for | Why |
|---|---|---|
| Location | The room or area, using the name on the signage | Lets a question naming a room reach the image |
| Objects, with counts | "Eleven chairs", never "some chairs" | Counting questions need this |
| Visible text, word for word | Every sign, poster and label, copied exactly | The Lv5 example asks for three exact words |
| Spatial relations | An LED wall on the north side, facing the entrance | Lv5 asks about arrangement |
| Colours and materials | Only where they identify something specific | Disambiguates similar rooms |

**Field 3 is the one to get right.** Tell the model to copy text exactly
rather than summarise, and to write `[unreadable]` rather than guess.

In [ ]:
# Put your images in an images/ folder next to this notebook.
# You need at least 5. Any mix of .jpg and .png is fine.
# At least one must have legible text on a sign, poster or wall: the
# transcription field is what this lab is really about.
IMAGES = sorted(str(p) for p in Path("images").iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert len(IMAGES) >= 5, (
    f"Found {len(IMAGES)} images in images/. This lab needs at least 5."
)
# TODO write three questions about YOUR images. One should need a count,
# one should need text copied off a sign, one should need layout.
TEST_QUESTIONS = [
    "",   # e.g. "How many tables are in Makerspace A?"
    "",   # e.g. "What three words are on the wall behind the seating?"
    "",   # e.g. "What is on the bench by the window?"
]
assert all(TEST_QUESTIONS), "Write your three test questions first."
print(len(IMAGES), "images")
for p in IMAGES:
    print("  ", Path(p).name)

5 images
   Brainstorming aerial.JPG
   Makerspace close up.JPG
   Open Event .JPG
   Open event.JPG
   makerspace.JPG


---
## Step 1: Write your prompt

Write instructions, not a request for a description. Compare the two
below before writing your own.

In [7]:
PROMPT_A = "Describe this image."

PROMPT_B = """List, in this order:
1. the room or area name shown on any sign
2. every object present, with a count
3. all visible text, copied exactly as written
4. what faces what, and what is next to what
5. colours and materials, only where distinctive

Write [unreadable] for text you cannot make out.
Do not summarise. Do not add anything not visible in the image."""

# ---- your prompt -------------------------------------------------
DESCRIPTION_PROMPT = """
TODO: write yours here.
"""

---
## Step 2: Run it and read the output

Before indexing anything.

In [10]:
print("PROMPT A\n" + "-" * 60)
print(describe_image(IMAGES[0], PROMPT_A))
print("\n\nPROMPT B\n" + "-" * 60)
print(describe_image(IMAGES[0], PROMPT_B))
print("\n\nSame model, same image, same cost.")

PROMPT A
------------------------------------------------------------
The photo shows a modern indoor common area or informal presentation space. In the foreground and left are low, modular cube seats arranged in neat rows — bright red, orange and yellow ottomans. To the right is a stepped, tiered seating structure with light wood faces and blue/teal cushions that can be used as bleacher-style seating. Behind the tiers a greenish upper wall houses a blue angular sign that reads "ENGINEERING INNOVATION." Along the back wall there are light wood cabinets and a small sink/dispensing station with posted notices. To the far right are glass doors leading into another room, and to the far left a corridor lined with more signs and shelving. The space has a carpeted floor with linear patterning and ceiling-mounted rectangular lights.


PROMPT B
------------------------------------------------------------
1) Room/area name shown on a sign
- ENGINEERING INNOVATION

2) Every object present, with a

In [11]:
descriptions = {}
for path in IMAGES:
    descriptions[path] = describe_image(path, DESCRIPTION_PROMPT)
    print("=" * 70)
    print(path)
    print("=" * 70)
    print(descriptions[path], "\n")

print("Can you answer each of these from the text above alone?")
for q in TEST_QUESTIONS:
    print("  -", q)

images\Brainstorming aerial.JPG
Here are several short writing options you can use — pick the one that fits your need:

1) Alt text (for accessibility)
A modern indoor collaborative space with colorful modular cube seats in red, orange and yellow, tiered wooden seating steps labeled “ENGINEERING INNOVATION,” carpeted floor, and glass doors to the right.

2) Short caption
Bright, flexible seating for engineering innovation.

3) Instagram / social post
New collaborative zone: colorful modular seating and tiered steps designed for group work, talks, and creative brainstorming. Come plug in, meet, and build something great. #Engineering #Innovation #Collaboration

4) Headline for signage or brochure
Engineering Innovation Hub — Flexible Spaces for Creative Work

5) Detailed description (for a site listing or accessibility page)
A contemporary indoor hub featuring modular cube seating in warm colors (red, orange, yellow) arranged in rows, adjacent to multi-level wooden tiered benches with c

---
## Step 3: Revise

Most teams need two or three attempts. The first is almost always too
general. Edit `DESCRIPTION_PROMPT` and re-run.

| Prompts that scored | Prompts that did not |
|---|---|
| Asked for counts explicitly | Asked to "describe this image" |
| Asked for text to be copied exactly | Asked for a summary or caption |
| Named the room or area | Did not mention text at all |
| Said what to do when unreadable | Left the model to decide what mattered |

---
## Step 4: Cache, index, re-score

**Cache by image path.** Without it you re-describe every image each
time you change something downstream, and experimenting becomes slow
enough that you stop.

Descriptions go into the same store as your scraped text, as ordinary
chunks. No separate pipeline. Tag them `kind="image"` so lab 5 can
filter on it.

In [14]:
Path("data").mkdir(exist_ok=True)
Path("data/descriptions.json").write_text(json.dumps(descriptions, indent=1))
print(f"cached {len(descriptions)} descriptions")

store = get_store("data/chroma", name="workshop")   # same index as lab 2
add_to_store(
    store,
    texts=list(descriptions.values()),
    metadatas=[{"url": p, "kind": "image"} for p in descriptions],
    ids=[f"img_{i}" for i in range(len(descriptions))],
)
print(f"indexed. store now holds {store.count()} chunks")

cached 5 descriptions
indexed. store now holds 36 chunks


In [15]:
for q in TEST_QUESTIONS:
    print("=" * 70)
    print("Q:", q)
    print("=" * 70)
    show(query(store, q, k=3), chars=220)

Q: How many tables are in Makerspace A?
  0.830  about-using-the-centre.txt
         9am. Teams of three or more may book Makerspace A; individuals should use the drop-in desks in Makerspace B. Bookings not claimed within twenty minutes are released to whoever is waiting. Consumables and storage Common c

  0.866  venues-makerspace-a.txt
         Home About Venues Equipment Funding Programmes Events News Contact Harbour Innovation Centre Faculty of Engineering Makerspace A Makerspace A is the largest open workspace in the Centre, on the second floor of the Kwok B

  0.944  venues-makerspace-b.txt
         Home About Venues Equipment Funding Programmes Events News Contact Harbour Innovation Centre Faculty of Engineering Makerspace B Makerspace B is the quieter of the two makerspaces, on the second floor next to the stairwe

Q: What three words are on the wall behind the seating?
  1.061  images\Brainstorming aerial.JPG
         Here are several short writing options you can use — pick t

---
## Step 5: Swap with the team next to you

Run their prompt over the same five images. Which questions does each
prompt answer that the other misses?

Two teams with different prompts get different Lv3 scores from identical
images. The prompt is the whole difference.

### Going further

- **Two prompts per image.** One for objects and layout, one for text
  only. The model attends to whatever you emphasise.
- **Split large images.** A poster downscaled to fit the input loses its
  small print. Cut it into overlapping tiles and describe each.
- **Run OCR alongside.** A vision model paraphrases; an OCR engine
  transcribes character by character. Tesseract and PaddleOCR both run
  locally, so both are permitted.